# DX 704 Week 1 Project

This week's project will build a portfolio risk and return model, and make investing recommendations for hypothetical clients.
You will collect historical data, estimate returns and risks, construct efficient frontier portfolios, and sanity check the certainty of the maximum return portfolio.

The full project description and a template notebook are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-01


Feel free to use optimization tools or libraries (such as CVXOPT or scipy.optimize) to perform any calculations required for this mini project.

### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Data

Collect historical monthly price data for the last 24 months covering 6 different stocks.
The data should cover 24 consecutive months including the last month that ended before this week's material was released on Blackboard.
To be clear, if a month ends between the Blackboard release and submitting your project, you do not need to add that month.

The six different stocks must include AAPL, SPY and TSLA.
At least one of the remaining 3 tickers must start with the same letter as your last name (e.g. professor Considine could use COIN).
This is to encourage diversity in what stocks you analyze; if you discuss this project with classmates, please make sure that you pick different tickers to differentiate your work.
Do not pick stocks with fewer than 24 consecutive months of price data.

In [43]:
import pandas as pd
from datetime import date

In [45]:
!uv pip install yfinance

Using Python 3.12.13 environment at: /Users/calebhadley/Projects/Fall_2026/AI_in_the_Field_Coursework/.venv
Checked 1 package in 31ms


In [46]:
import yfinance as yf
import pandas as pd

# disable the yfinance cache to prevent the 'database is locked' error
yf.set_tz_cache_location(None)

# HADLEY -> 
# A -> ABNB
# D -> DDOG
# Also picking a random one i am interested in -> COF
tickers = ['AAPL', 'SPY', 'TSLA', 'ABNB', 'DDOG', 'COF'] # Apple, S&P 500, Tesla, Airbnb, Datadog, Capital One

# September 2024 to August 2026 inclusive (24 mnths)
start_date = '2024-09-01'
end_date = '2026-09-01' 

print(f"Downloading historical daily data for: {tickers}")

# auto_adjust=True is the default, meaning 'Close' IS the adjusted close price
raw_daily_data = yf.download(tickers, start=start_date, end=end_date)
raw_daily_data

[**********************83%***************        ]  5 of 6 completed

[*********************100%***********************]  6 of 6 completed


Price            Close                                                  \
Ticker            AAPL        ABNB         COF        DDOG         SPY   
Date                                                                     
2024-09-03  220.921951  114.980003  141.644913  111.480003  539.312622   
2024-09-04  219.017883  115.209999  139.864639  108.650002  538.208862   
2024-09-05  220.535187  116.160004  138.230255  110.089996  536.899780   
2024-09-06  218.988129  114.279999  135.224197  107.199997  527.863647   
2024-09-09  219.077393  116.360001  138.726440  107.699997  533.773682   
...                ...         ...         ...         ...         ...   
2026-08-25  309.899994  190.500000  216.240005  222.990005  765.909973   
2026-08-26  313.450012  188.070007  217.210007  227.669998  766.080017   
2026-08-27  314.579987  184.399994  216.669998  242.929993  771.099976   
2026-08-28  319.700012  189.429993  215.669998  236.979996  769.349976   
2026-08-31  316.850006  183.220001  214.509995  237.039993  767.049988   

Price                         High                                      ...  \
Ticker            TSLA        AAPL        ABNB         COF        DDOG  ...   
Date                                                                    ...   
2024-09-03  210.600006  227.100264  117.834999  143.269547  115.500999  ...   
2024-09-04  219.410004  219.940161  115.989998  143.172276  111.400002  ...   
2024-09-05  230.169998  223.609461  116.449997  141.421156  110.500000  ...   
2024-09-06  210.729996  223.371459  117.260002  141.139042  110.969002  ...   
2024-09-09  216.270004  219.434407  117.919998  139.631171  109.019997  ...   
...                ...         ...         ...         ...         ...  ...   
2026-08-25  350.250000  313.589996  191.360001  218.149994  229.080002  ...   
2026-08-26  345.820007  315.429993  192.429993  218.250000  234.949997  ...   
2026-08-27  354.809998  315.399994  188.718994  218.690002  251.990005  ...   
2026-08-28  348.750000  322.369995  191.490005  219.149994  247.875000  ...   
2026-08-31  367.950012  321.239990  189.330002  215.440002  239.699997  ...   

Price             Open                                        Volume           \
Ticker             COF        DDOG         SPY        TSLA      AAPL     ABNB   
Date                                                                            
2024-09-03  141.518439  114.870003  547.508550  215.259995  50190600  4678800   
2024-09-04  142.345374  110.769997  537.476207  210.589996  43840200  3281600   
2024-09-05  140.856909  108.010002  538.150208  223.490005  36615400  3593400   
2024-09-06  138.239995  109.870003  537.222118  232.600006  48423000  4072400   
2024-09-09  136.605650  108.269997  532.054434  216.199997  67180000  4178600   
...                ...         ...         ...         ...       ...      ...   
2026-08-25  217.889999  226.889999  766.159973  349.579987  25869800  4504500   
2026-08-26  216.240005  220.460007  764.729980  345.269989  34024500  3369500   
2026-08-27  215.970001  237.000000  768.500000  346.160004  32419200  4792500   
2026-08-28  217.320007  240.220001  771.760010  357.100006  38649400  4829400   
2026-08-31  214.250000  233.964996  767.330017  347.209991  41242700  7598300   

Price                                              
Ticker          COF     DDOG       SPY       TSLA  
Date                                               
2024-09-03  1530900  3355800  60600100   76714200  
2024-09-04  1996400  3536800  47224900   80651800  
2024-09-05  2261400  2701600  44264300  119355000  
2024-09-06  2381600  3311300  68493800  112177000  
2024-09-09  1998400  2720500  40445800   67443500  
...             ...      ...       ...        ...  
2026-08-25  2164600  3091400  27422300   29816700  
2026-08-26  2080900  4044000  28751600   28573000  
2026-08-27  2608800  4550400  34557100   30136200  
2026-08-28  2537600  2519300  36744300   32972200  
2026-08-31  3906300  4240800  38810800   6173

In [47]:
daily_close = raw_daily_data['Close']
daily_close

Ticker,AAPL,ABNB,COF,DDOG,SPY,TSLA
Date,,,,,,
2024-09-03,220.921951,114.980003,141.644913,111.480003,539.312622,210.600006
2024-09-04,219.017883,115.209999,139.864639,108.650002,538.208862,219.410004
2024-09-05,220.535187,116.160004,138.230255,110.089996,536.899780,230.169998
2024-09-06,218.988129,114.279999,135.224197,107.199997,527.863647,210.729996
2024-09-09,219.077393,116.360001,138.726440,107.699997,533.773682,216.270004
...,...,...,...,...,...,...
2026-08-25,309.899994,190.500000,216.240005,222.990005,765.909973,350.250000
2026-08-26,313.450012,188.070007,217.210007,227.669998,766.080017,345.820007
2026-08-27,314.579987,184.399994,216.669998,242.929993,771.099976,354.809998


In [48]:
# Resample the daily data to the last trading day of each month
# 'ME' stands for Month End, and .last() captures the final trading session price
monthly_prices = daily_close.resample('ME').last()
monthly_prices

Ticker,AAPL,ABNB,COF,DDOG,SPY,TSLA
Date,,,,,,
2024-09-30,231.067093,126.809998,145.662720,115.059998,562.210449,261.630005
2024-10-31,224.035904,134.789993,158.367966,125.440002,557.193542,249.850006
2024-11-30,235.620117,136.110001,187.400284,152.750000,590.420898,345.160004
2024-12-31,248.615784,131.410004,174.038956,142.889999,576.215332,403.839996
2025-01-31,234.299667,131.169998,198.819382,142.710007,591.690369,404.600006
2025-02-28,240.361588,138.869995,196.317932,116.550003,584.178955,292.980011
2025-03-31,220.772079,119.459999,175.516357,99.209999,551.628906,259.160004
2025-04-30,211.200943,121.919998,176.456070,102.160004,546.846252,282.160004
2025-05-31,199.883942,129.000000,185.749542,117.879997,581.212769,346.459991


Save the data as a TSV file named "historical_prices.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
The date should be the last trading day of the month, so it may not be the last day of the month.
For example, the last trading day of November 2024 was 2024-11-29.
The remaining columns should contain the adjusted closing prices of the corresponding stock tickers on that day.


In [49]:
# YOUR CHANGES HERE

monthly_prices.to_csv("price_data.tsv", sep='\t')
monthly_prices.head()

Ticker,AAPL,ABNB,COF,DDOG,SPY,TSLA
Date,,,,,,
2024-09-30,231.067093,126.809998,145.662720,115.059998,562.210449,261.630005
2024-10-31,224.035904,134.789993,158.367966,125.440002,557.193542,249.850006
2024-11-30,235.620117,136.110001,187.400284,152.750000,590.420898,345.160004
2024-12-31,248.615784,131.410004,174.038956,142.889999,576.215332,403.839996
2025-01-31,234.299667,131.169998,198.819382,142.710007,591.690369,404.600006


Submit "historical_prices.tsv" in Gradescope.

## Part 2: Calculate Historical Asset Returns

Calculate the historical asset returns based on the price data that you previously collected.

In [50]:
# YOUR CHANGES HERE

...

Ellipsis

Save the data as a TSV file named "historical_returns.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
Each row should have the date at the end of the month and the corresponding *relative* price changes.
For example, if the previous price was \$100 and the new price is \$110, the return value should be 0.10.
There should only be 23 rows of data in this file, since they are computed as the differences of 24 prices.

In [51]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "historical_returns.tsv" in Gradescope.

## Part 3: Estimate Returns

Estimate the expected returns for each asset using the previously calculated return data.
Just compute the average (mean) return for each asset over your data set; do not use other estimators that have been mentioned.
This will serve as your estimate of expected return for each asset.

In [52]:
# YOUR CHANGES HERE

...

Ellipsis

Save the estimated returns in a TSV file named "estimated_returns.tsv" and include a header row with the column names "asset" and "estimated_return".

In [53]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "estimated_returns.tsv" in Gradescope.

## Part 4: Estimate Risk

Estimate the covariance matrix for the asset returns to understand how the assets move together.

In [54]:
# YOUR CHANGES HERE

...

Ellipsis

Save the estimated covariances to a TSV file named "estimated_covariance.tsv".
The header row should have a blank column name followed by the names of the assets.
Each data row should start with the name of an asset for that row, and be followed by the individual covariances corresponding to that row and column's assets.
(This is the format of pandas's `to_csv` method with `sep="\t"` when used on a covariance matrix as computed in the examples.)

In [55]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "estimated_covariance.tsv" in Gradescope.

## Part 5: Construct the Maximum Return Portfolio

Compute the maximum return portfolio based on your previously estimated risks and returns.

In [56]:
# YOUR CHANGES HERE

...

Ellipsis

Save the maximum return portfolio in a TSV file named "maximum_return.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [57]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "maximum_return.tsv" in Gradescope.

## Part 6: Construct the Minimum Risk Portfolio

Compute the minimum risk portfolio based on your previously estimated risks.

In [58]:
# YOUR CHANGES HERE

...

Ellipsis

Save the minimum risk portfolio in a TSV file named "minimum_risk.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [59]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "minimum_risk.tsv" in Gradescope.

## Part 7: Build Efficient Frontier Portfolios

Compute 101 portfolios along the mean-variance efficient frontier with evenly spaced estimated returns.
The first portfolio should be the minimum risk portfolio from part 4, and the last portfolio should be the maximum return portfolio from part 3.
The estimated return of each portfolio should be higher than the previous by one percent of the difference between the first and last portfolios.
That is, the estimated return of the portfolios should be similar to `np.linspace(min_risk_return, max_return, 101)`.


In [60]:
# YOUR CHANGES HERE

...

Ellipsis

Save the portfolios in a TSV file named "efficient_frontier.tsv".
The header row should have columns "index", "return", "risk", and all the asset tickers.
Each data row should have the portfolio index (0-100), the estimated return of the portfolio, the estimated standard deviation (not variance) of the portfolio, and all the asset allocations (which should sum to one).

In [61]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "efficient_frontier.tsv" in Gradescope.

## Part 8: Check Maximum Return Portfolio Stability

Check the stability of the maximum return portfolio by resampling the estimated risk/return model.

Repeat 1000 times -
1. Use `np.random.multivariate_normal` to generate 23 return samples using your previously estimated risks and returns.
2. Estimate the return of each asset using that resampled return history.
3. Check which asset had the highest return in those resampled estimates.

This procedure is a reduced and simplified version of the Michaud resampled efficient frontier procedure that takes uncertainty in the risk model into account.

In [62]:
# YOUR CHANGES HERE

...

Ellipsis

Save a file "max_return_probabilities.tsv" with the distribution of highest return assets.
The header row should have columns "asset" and "probability".
There should be a data row for each asset and its sample probability of having the highest return based on those 1000 resampled estimates.


In [63]:
# YOUR CHANGES HERE

...

Ellipsis

Submit "max_return_probabilities.tsv" in Gradescope.

## Part 9: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 10: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.